# Feature Engineering & Modelling Template

The next step after your dataset passes inspection (`02_dataset_inspection_template.ipynb`). This notebook covers Steps 4-6 of the machine learning workflow from *Machine Learning Part I*: feature engineering, modelling, and evaluation.

**How to use this notebook:**
1. Copy this file and rename it for your project.
2. Carry over the same `FILE_PATH`, `FILE_FORMAT`, `TARGET_COLUMN`, `PROBLEM_TYPE`, `READ_KWARGS` you used in notebook 02 (Section 0 below), plus the cleaning decisions you made there.
3. Fill in the **cleaning cell** in Section 1 with whatever fixes notebook 02 told you your dataset needed (drop columns, fill missing values, fix dtypes).
4. Run every cell top to bottom.
5. `PROBLEM_TYPE` (`"classification"` or `"regression"`) controls everything downstream: which model is used, which metrics are printed, and how results are plotted. You don't need to change any code because of it.

**What this notebook assumes:** you already ran notebook 02 and know your dataset's shape, missing values, duplicates, and target column. This notebook does not re-run those checks -- it builds on them.

## 0. Setup and Config

Same config pattern as notebook 02 -- copy your values across.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("Environment ready.")

In [ ]:
# =========================== EDIT THIS CELL ===========================
# Use the SAME values you settled on in notebook 02.

FILE_PATH = "data/sample_student_dataset.csv"
FILE_FORMAT = "csv"          # "csv", "excel", or "json"
TARGET_COLUMN = "churn"      # the column you're predicting
PROBLEM_TYPE = "classification"   # "classification" or "regression"
READ_KWARGS = {}

# Columns to drop before modelling: IDs, free text, anything that leaked out of
# notebook 02's inspection (e.g. a column suspiciously correlated with the target,
# or one only known after the outcome happens -- see Part I slide 26, "Data leakage").
COLUMNS_TO_DROP = []

# ========================================================================

readers = {"csv": pd.read_csv, "excel": pd.read_excel, "json": pd.read_json}
df = readers[FILE_FORMAT](FILE_PATH, **READ_KWARGS)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target: {TARGET_COLUMN!r}  |  Problem type: {PROBLEM_TYPE!r}  |  Dropping: {COLUMNS_TO_DROP or 'nothing'}")

## 1. Apply the Cleaning Decisions From Notebook 02

Notebook 02 only diagnosed problems -- it never modified `df`. This is where you actually apply the fixes it told you to make. Edit the cell below to match what you decided: drop the columns in `COLUMNS_TO_DROP`, remove duplicate rows, and fix any dtypes notebook 02 flagged (e.g. numbers stored as text, or dates stored as strings).

Leave the imputing (filling missing values) for Section 2 -- that belongs with feature engineering, not cleaning, because *how* you impute is itself a modelling choice.

In [ ]:
before_rows = len(df)

# Drop columns you decided don't belong (IDs, leakage, free text)
df = df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])

# Drop exact duplicate rows (do this BEFORE splitting, never after -- see Part II slide 26)
df = df.drop_duplicates().reset_index(drop=True)

# ---- Add any dtype fixes notebook 02 flagged, for example: ----
# df["signup_date"] = pd.to_datetime(df["signup_date"])
# df["price"] = df["price"].str.replace("$", "").str.replace(",", "").astype(float)

print(f"Rows: {before_rows} -> {len(df)} after dropping duplicates")
print(f"Columns: {df.shape[1]}")
df.head()

## 2. Feature Engineering

Separate `X` (features) from `y` (target), encode categories into numbers, and split into train/test **before** fitting anything -- including the scaler. Fitting a scaler or imputer on the full dataset leaks test information into training, a silent, common bug (Part II, slide 20).

Order matters:
1. Split `X` / `y` off the target.
2. `train_test_split` -- do this first.
3. Fit the imputer and scaler on **training data only**, then apply (`transform`) to both train and test.

In [ ]:
assert TARGET_COLUMN in df.columns, f"{TARGET_COLUMN!r} not found -- check the config cell."

# Rows with a missing target can't be used for supervised learning -- there's nothing to
# learn from a row where the "correct answer" is unknown, and you can't impute a target
# the way you can impute a feature. Drop them here rather than letting .fit() crash later.
n_missing_target = df[TARGET_COLUMN].isna().sum()
if n_missing_target > 0:
    print(f"Dropping {n_missing_target} rows with a missing {TARGET_COLUMN!r} (can't train on an unknown answer).")
    df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

# Encode categorical columns into numeric 0/1 columns (models need numbers, not text)
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if categorical_cols:
    print(f"One-hot encoding: {categorical_cols}")
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
else:
    print("No categorical columns to encode.")

# If the target itself is text (e.g. "yes"/"no", "spam"/"ham"), encode it to 0/1/2...
if PROBLEM_TYPE == "classification" and y.dtype == "object":
    y, label_names = pd.factorize(y)
    print(f"Target encoded: {dict(enumerate(label_names))}")

print(f"\nFinal feature matrix: {X.shape[0]} rows x {X.shape[1]} columns")
X.head()

In [ ]:
# Split BEFORE fitting anything. stratify keeps class proportions intact for classification.
stratify = y if PROBLEM_TYPE == "classification" else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
)

# Impute missing values -- fit on TRAINING data only, then apply to both splits
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# Scale numeric features -- same rule: fit on train, transform both
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print("Imputer and scaler were fit on training data only -- test set is still unseen.")

## 3. Modelling

`RandomForest` is used as a strong, low-maintenance default for both problem types -- it handles non-linear relationships, doesn't need much tuning to get a reasonable first result, and (Part I, slide 18) picking *an* algorithm and calling `fit()` on the training set only is the point of this step, not finding the perfect one on the first try.

Swapping it for another scikit-learn estimator later is a one-line change (Part II, slide 19) -- everything below keeps working as long as the new model also has `.fit()` / `.predict()`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

if PROBLEM_TYPE == "classification":
    model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
elif PROBLEM_TYPE == "regression":
    model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
else:
    raise ValueError(f"PROBLEM_TYPE must be 'classification' or 'regression', got {PROBLEM_TYPE!r}")

model.fit(X_train, y_train)  # test set is untouched until Section 4

print(f"Trained {type(model).__name__} on {len(X_train)} rows.")
print(f"Held out {len(X_test)} rows for evaluation -- not used yet.")

## 4. Evaluation

Now the held-out test set finally gets used. Which metrics matter depends on `PROBLEM_TYPE` (Part I, slide 14):

- **Classification** -- Accuracy, Precision, Recall, F1, plus a confusion matrix.
- **Regression** -- MAE, RMSE, R², plus a predicted-vs-actual scatter plot.

The cell below prints the right set automatically.

In [ ]:
y_pred = model.predict(X_test)

if PROBLEM_TYPE == "classification":
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    )

    average = "binary" if len(np.unique(y)) == 2 else "weighted"
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}  (how often am I right overall?)")
    print(f"Precision: {precision_score(y_test, y_pred, average=average):.3f}  (when I predict positive, how often am I right?)")
    print(f"Recall:    {recall_score(y_test, y_pred, average=average):.3f}  (of all real positives, how many did I catch?)")
    print(f"F1:        {f1_score(y_test, y_pred, average=average):.3f}  (balance of precision & recall)")

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots()
    im = ax.imshow(cm, cmap="Blues")
    labels = sorted(np.unique(y))
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion matrix")
    fig.colorbar(im)
    plt.show()

else:  # regression
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)
    print(f"MAE:  {mae:.3f}  (average absolute error, in your target's units)")
    print(f"RMSE: {rmse:.3f}  (penalises large errors more than MAE)")
    print(f"R^2:  {r2:.3f}  (share of variance explained; 1.0 = perfect fit)")

    fig, ax = plt.subplots()
    ax.scatter(y_test, y_pred, alpha=0.6)
    lims = [min(np.min(y_test), y_pred.min()), max(np.max(y_test), y_pred.max())]
    ax.plot(lims, lims, "r--", label="perfect prediction")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title("Predicted vs. actual")
    ax.legend()
    plt.show()

### Cross-validation and hyperparameter tuning

One train/test split can be lucky or unlucky. `cross_val_score` splits the data 5 different ways and reports all 5 scores plus their mean, giving a more reliable estimate than a single split. `GridSearchCV` then automates trying different hyperparameter combinations to find the best-scoring one -- instead of guessing values by hand.

In [ ]:
scoring = "f1" if PROBLEM_TYPE == "classification" and len(np.unique(y)) == 2 else (
    "f1_weighted" if PROBLEM_TYPE == "classification" else "r2"
)

# Use the full feature matrix here (imputed/scaled the same way, fit fresh each fold internally
# by cross_val_score) -- for simplicity this template reuses X_train, which is fine for a
# first pass; a Pipeline (Part II, slide 20) is the more rigorous way to do this end-to-end.
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring=scoring)
print(f"5-fold cross-validated {scoring}: {cv_scores.round(3)} -> mean {cv_scores.mean():.3f}")

param_grid = {"max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 5, 10]}
grid = GridSearchCV(
    type(model)(n_estimators=200, random_state=RANDOM_STATE),
    param_grid, scoring=scoring, cv=3,
)
grid.fit(X_train, y_train)
print(f"\nBest hyperparameters: {grid.best_params_}")
print(f"Best cross-val {scoring}: {grid.best_score_:.3f}")

final_model = grid.best_estimator_

### Which features actually mattered?

Random forests can report how much each feature contributed to their predictions. This is worth checking against your own intuition -- if a feature you expected to matter shows near-zero importance, or an unexpected one dominates, that's often the first clue toward either a data quality issue (see notebook 02's leakage check) or a genuine insight about your problem.

In [ ]:
importances = pd.Series(final_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(importances))))
importances.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Feature importance (top 15)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

print("If one feature dominates far above the rest, double check it isn't leakage")
print("(Part I slide 26) -- would you actually have this value at prediction time?")

## Summary & Next Steps

You just ran Steps 4-6 of the ML workflow:

1. **Cleaning applied** (Section 1) -- the fixes notebook 02 diagnosed, actually made.
2. **Feature engineering** (Section 2) -- encoded categories, split before fitting anything, imputed and scaled on training data only.
3. **Modelling** (Section 3) -- picked an algorithm based on `PROBLEM_TYPE`, fit on the training set.
4. **Evaluation** (Section 4) -- scored on the held-out test set with the right metrics, cross-validated, tuned hyperparameters, checked feature importance.

**If your scores look good:** you have a working first model. From here, the loop (Part I, slide 18) is normal -- go back to Section 2 and try different features, or Section 3 and try a different algorithm, and see if scores improve.

**If your scores look too good** (e.g. near-perfect accuracy or R² on your first try): be suspicious before celebrating. Revisit the feature importance plot and notebook 02's leakage check -- a feature that secretly contains the answer produces exactly this pattern (Part I, slide 26).

**If your scores look poor:** that's useful information, not failure. Consider: do you have enough rows? Is the target too noisy to predict from these features? Would a different algorithm help? Loop back to notebook 02 if you suspect the dataset itself, or to Section 2 here if you think better features would help.

**What's not covered here:** Step 7, deployment and monitoring -- saving the trained model and watching for drift once it's in use (see `01_introduction_to_machine_learning.ipynb`, Section 5, for a worked example of that step).